In [32]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables BEFORE importing LangChain

# Initialize LangSmith tracing
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import HumanMessage,ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langchain.tools import tool,ToolRuntime
from scripts import base_tools
from langgraph.types import Command
from pathlib import Path
from langsmith import Client
from dataclasses import dataclass
from langgraph.checkpoint.postgres import PostgresSaver
import psycopg
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# Initialize LangSmith client
ls_client = Client()

In [19]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [33]:
system_message = "You are a helpful assistant that provides information about the user when asked."


Short Term Memory using sqlite

In [34]:
config={"configurable":{"thread_id":"user_thread_1"}}

In [ ]:
def get_agent():
    
    os.makedirs("db", exist_ok=True)
    # checkpointer_saver= InMemorySaver()
    conn = sqlite3.connect("db/02_agent_checkpoints.db",check_same_thread=False)
    checkpointer_saver = SqliteSaver(conn)
    checkpointer_saver.setup()
    agent=create_agent(model=model,system_prompt=system_message, tools=[base_tools.get_weather,base_tools.web_search],checkpointer=checkpointer_saver)
    return agent


In [23]:
agent=get_agent()
agent.invoke({'messages': [HumanMessage(content="my name is gowish?")]}, config=config)
response = agent.invoke({'messages': [HumanMessage(content="what is my name?")]}, config=config)
print(response)

{'messages': [HumanMessage(content='my name is gowish?', additional_kwargs={}, response_metadata={}, id='d111b32d-5a95-4411-b608-fe5271a77c19'), AIMessage(content=[{'type': 'text', 'text': "It's nice to meet you, Gowish! Is there anything I can help you with today?", 'extras': {'signature': 'CqwBAQw51sfpzWvIUkrk6e5YDkF0LEAOfO8mZwm5AYgN+HT9TKNcc+w5pPGnOMVp+swQ47LrCE7OmPs22r5BFtDnkiO59LScp234Trm57hsC5dI/ODAG1ppvZM2XCWwo7T9FkWmgMmo1sLnpbjgsOjkr7HJvD8e9Aw4nufbZPmrcurC0alUtEroIBrekz/v+oD6ZbynDXeERwgVIqWCh+1MSsOwKwTTLO9RqmDczyg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e59a0-1f7a-7192-abd5-2f7235f59cb9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 51, 'total_tokens': 274, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 30}}), HumanMessage(content='what is my name?', additional

Short Term Memory using Postgresql DB

In [35]:
config={"configurable":{"thread_id":"user_thread_1"}}

In [24]:

def get_agent():
    conn = psycopg.connect(os.getenv("POSTGRESQL_API_KEY"),autocommit=True)
    pg_checkpointer_saver=PostgresSaver(conn)
    pg_checkpointer_saver.setup()
    agent=create_agent(model=model,system_prompt=system_message, tools=[base_tools.get_weather,base_tools.web_search],checkpointer=pg_checkpointer_saver)
    return agent


In [25]:
agent=get_agent()
agent.invoke({'messages': [HumanMessage(content="my name is gowish?")]}, config=config)
response = agent.invoke({'messages': [HumanMessage(content="what is my name?")]}, config=config)
print(response)

{'messages': [HumanMessage(content='my name is gowish?', additional_kwargs={}, response_metadata={}, id='13bc35c2-e7e8-4df6-900e-8820c75b8035'), AIMessage(content=[{'type': 'text', 'text': "It's nice to meet you, Gowish!", 'extras': {'signature': 'CqwBAQw51seVzPmORITnuANADlwbbQvaNjHzbN0cCV5VK5e4GqvJyxGuykIugsJGjCyALZGrGzbYMXMEJpz1hdkQCPDUkHl7eknNAzMU5iZjKrL1kGa6zvsb0u7Hk0FmiyjS2IvDIPMjYT8/9pQCHrPvW91dl1/vgNhLh10eKulXK4emt1MUh76Rab6yDXvoomF8gz1xwAFQCtWY47vvKFpYPmy9JMzG/noFnP5UQw=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e59f9-81b6-7b03-885a-1a8dffd7e3d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 41, 'total_tokens': 264, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 30}}), HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id='bc9fafc

Context offloading

In [37]:
def summarize_conversation(summary:str,runtime:ToolRuntime):
    """Simple conversation summary to disk for context offloading"""
    user_id=runtime.context.user_id
    thread_id=runtime.context.thread_id
    summary_dir=Path(f"data/{user_id}/{thread_id}")
    summary_dir.mkdir(parents=True,exist_ok=True)
    summary_path=summary_dir / "conversation_summary.txt"
    summary_path.write_text(summary)
    return f"Conversation summary saved to {summary_path}"

In [38]:
@dataclass
class UserContext:
    user_id: str
    thread_id: str

In [39]:

def get_agent():
    conn = psycopg.connect(os.getenv("POSTGRESQL_API_KEY"),autocommit=True)
    pg_checkpointer_saver=PostgresSaver(conn)
    pg_checkpointer_saver.setup()
    agent=create_agent(model=model,system_prompt=system_message, tools=[summarize_conversation],checkpointer=pg_checkpointer_saver,context_schema=UserContext)
    return agent


In [40]:
config={"configurable":{"thread_id":"user_thread_2"}}
user_context = UserContext(user_id="user_123", thread_id="user_thread_2")

In [41]:
agent=get_agent()
agent.invoke({'messages': [HumanMessage(content="can u please help me with the good crickerter in india team")]}, config=config,context=user_context)

{'messages': [HumanMessage(content='can u please help me with the good crickerter in india team', additional_kwargs={}, response_metadata={}, id='83ae8a6c-213a-4e22-9ebb-82897e276b39'),
  AIMessage(content=[{'type': 'text', 'text': 'Cricket is a team sport, and what makes a player "good" can depend on many factors like their batting, bowling, fielding, leadership, and performance in different formats of the game. Also, "good" can be subjective and depend on individual preferences!\n\nCould you tell me what specific aspects you\'re interested in? For example, are you looking for a good batsman, bowler, all-rounder, or perhaps a player known for their captaincy?', 'extras': {'signature': 'CoEFAQw51scAokCkUsfw7nRxLkZOv0t9iFf+kBB/G6iexYQqdNznJT4MeEb45LYZbE7FupJWN7AMhwMwTv38T8o/DR5LnfMmH7SlCISWJAymrznjUhm5JF7lCsbGqN8gyoPDAVlt2We81rzjomkDRI3r+xzx7B9zHYqZzCQhc2fbX2NLXUva0uqmvKjufuxMxrj5OSlXa9cfabip+yt3e2X2gOFK5KG8t4+fknqe4MqHoDPzi0oBj1lBbAshPtA3ky5uqfxhXE7Yud7I8SDcAJry20kYcI3HhZ3mbR573XPsxK+3

In [42]:
response=agent.invoke({'messages': [HumanMessage(content="can u summarize all the conversation we have use summary tool")]}, config=config,context=user_context)
print(response)

c:\INTELLIPAT\PROJECTS\GEN_AI\agent-fundamentals\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user...read_id='user_thread_2'), input_type=UserContext])
  function=lambda v, h: h(v), schema=original_schema
c:\INTELLIPAT\PROJECTS\GEN_AI\agent-fundamentals\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user...read_id='user_thread_2'), input_type=UserContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='can u please help me with the good crickerter in india team', additional_kwargs={}, response_metadata={}, id='83ae8a6c-213a-4e22-9ebb-82897e276b39'), AIMessage(content=[{'type': 'text', 'text': 'Cricket is a team sport, and what makes a player "good" can depend on many factors like their batting, bowling, fielding, leadership, and performance in different formats of the game. Also, "good" can be subjective and depend on individual preferences!\n\nCould you tell me what specific aspects you\'re interested in? For example, are you looking for a good batsman, bowler, all-rounder, or perhaps a player known for their captaincy?', 'extras': {'signature': 'CoEFAQw51scAokCkUsfw7nRxLkZOv0t9iFf+kBB/G6iexYQqdNznJT4MeEb45LYZbE7FupJWN7AMhwMwTv38T8o/DR5LnfMmH7SlCISWJAymrznjUhm5JF7lCsbGqN8gyoPDAVlt2We81rzjomkDRI3r+xzx7B9zHYqZzCQhc2fbX2NLXUva0uqmvKjufuxMxrj5OSlXa9cfabip+yt3e2X2gOFK5KG8t4+fknqe4MqHoDPzi0oBj1lBbAshPtA3ky5uqfxhXE7Yud7I8SDcAJry20kYcI3HhZ3mbR573XPsxK+3cG